# 1 章节介绍

本教程《LongCat-Next 单卡推理实践》以美团开源的原生多模态大模型 **LongCat-Next** 为例，展示如何在**单张 Atlas A3 NPU** 上使用 cann-recipes-infer 离线推理框架部署一个完整权重约 151GB（BF16）的多模态 MoE 模型，完成文本、图像、语音多类任务的推理部署实践。

课程按 recipes 工作流展开：查看并修改 `models/longcat_next/config/` 下的 YAML 配置，通过 `executor/scripts/infer.sh` 拉起离线推理，再从 `res/` 结果目录与 `prof/` 性能产物中读取指标。

推理框架在课程中自动获取：第 2 章首次运行时会把官方 [cann-recipes-infer](https://gitcode.com/cann/cann-recipes-infer) 仓克隆到运行目录（全量约 300MB），并在其固定基线提交上应用课程的单卡推理 patch（上游仓内置的是多卡实现），后续章节与后续会话直接复用。

## LongCat-Next 模型简介

LongCat-Next 是一个原生多模态统一模型，文本、图像、语音在同一个 Transformer 主干中统一为离散 token 序列，覆盖六类任务：

| 任务 | 输入 | 产物 |
| --- | --- | --- |
| `text` 文本生成 | 文本 | `text.txt` |
| `image_understanding` 图像理解 | 文本 + 图像 | `text.txt` |
| `image_generation` 图像生成 | 文本 | `image*.png` |
| `audio_to_text` 语音理解 | 文本 + 音频 | `text.txt` |
| `speech_synthesis` 语音合成 | 参考音频 + 文本 | `audio*.wav` |
| `audio_to_audio` 语音复刻 | 音频 + 参考音频 | `audio*.wav` |

模型主干与生成头的关键结构（数值来自官方权重 `config.json`）：

| 组件 | 参数 |
| --- | --- |
| 主干层数 | 14 层，每层 `[MLA, shortcut-MoE, dense MLP, MLA, dense MLP]` |
| 隐藏维度 | 3072，注意力头 32 |
| MLA 低秩注意力 | `q_lora_rank=1536`、`kv_lora_rank=512`、`nope=128 + rope=64` |
| shortcut MoE | 256 个路由专家，`moe_topk=12`，专家 FFN 1024 |
| N-gram 词向量 | 邻居数 4、切分数 4，与稠密词表（28.2 万）协同 |
| 上下文长度 | 131072（128K） |
| 视觉编码 | Qwen2.5-VL 风格 vision tower（hidden 1280） |
| 音频编码 | Whisper 风格 encoder（32 层、128 mel bins） |
| 图像生成头 | 深度 Transformer（4 层）+ image_decoder（扩散 refiner） |
| 语音生成头 | 深度 Transformer（4 层）+ cosy24k vocoder（24 kHz） |

## 单卡推理的挑战：为什么需要 CPU Offload

LongCat-Next 完整 BF16 权重约 **151GB**，而单张 Atlas A3 的 HBM 只有 **64GB**——权重无法全部驻留设备。本样例的解法是 **CPU offload**：

- 主干稠密权重（attention 投影、dense MLP、embedding 输出层）常驻 NPU；
- **MoE 专家权重**与 **N-gram 嵌入**通过 mmap 驻留 CPU 内存，推理时按需分块搬运到 NPU 计算；
- 框架在配置层强制这一约束：`world_size=1` 时必须同时开启 `enable_moe_cpu_offload` 与 `enable_ngram_cpu_offload`，否则直接报错（框架在配置校验阶段即拒绝）。

代价是每个 decode step 需要等待 CPU 专家计算与 H2D 搬运，单卡吞吐明显低于多卡 TP 部署；其归因（区分设备算子耗时与 CPU offload 耗时）可借助框架的 `enable_profiler` 开关采集 `prof/` 产物自行分析。

## 学习目标

- 了解多模态 MoE 大模型在昇腾 NPU 上的部署流程：环境准备、权重准备、YAML 配置与 `infer.sh` 启动。
- 掌握请求清单（request manifest）驱动的多任务推理：任务类型、素材路径、采样参数如何配置。

## 章节安排

- [1 章节介绍](./01_chapter_intro.ipynb)
- [2 Baseline 冒烟跑通](./02_baseline_inference.ipynb)
- [3 文本生成体验](./03_text_generation.ipynb)
- [4 多模态任务：图理解、语音与图像生成](./04_multimodal_tasks.ipynb)

## 软硬件与数据要求

| 项目 | 要求 |
| --- | --- |
| 支持硬件 | Atlas A3 训练/推理系列产品（单卡 64GB HBM） |
| CANN | ≥ 8.5.1（教程在 8.5.1 上验证） |
| Python | 3.12（配套 conda 环境 `cann-recipes-longcat`） |
| PyTorch | `torch==2.8.0`、`torch_npu==2.8.0.post4` |
| 主机内存 | ≥ 64GB 可用（权重 mmap 与 CPU 专家计算，推荐 ≥ 128GB） |
| 磁盘 | 下载目录 ≥ 160GB 可用（权重约 151GB）+ 推理仓克隆约 300MB + 结果目录若干 GB |
| 模型权重 | `meituan-longcat/LongCat-Next`，第 2 章首次运行自动从 ModelScope 下载（支持断点续传）；已有本地权重可设 `LONGCAT_NEXT_MODEL_PATH` 跳过 |
| 演示素材 | 官方 `assets/`（book.png、math1.wav、system_audio.wav 等），随权重目录提供 |

> **注意**：
> - 本教程权重体积大（151GB）：首次运行第 2 章会自动从 ModelScope 下载（视带宽约 25 分钟~4 小时，支持断点续传）；已有本地权重的，设置 `LONGCAT_NEXT_MODEL_PATH` 后直接使用。
> - 每章的推理运行需要数分钟到二十分钟不等（含约 2-3 分钟权重加载），请在空闲 NPU 上执行。
